# Phase 2 — penalize the top eigenvalue, or the whole trace?

<a href="https://colab.research.google.com/github/fatemehsalmani75/active-subspace-bloodmnist/blob/main/notebooks/phase2_spectral_regularization.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Phase 1: across six models, universal fooling followed $\lambda_1(C)$ (Spearman 0.886), but $\operatorname{tr} C$ followed it just as
well. This notebook separates the two by training with a penalty on each.

For a batch with input gradients $g_i = \nabla_x \ell(x_i)$:

- **$\lambda_1$ penalty**: $R = \frac{1}{B}\sum_i (g_i^\top u)^2$, with $u$ the top eigenvector of the batch covariance, tracked by one
  power-iteration step per batch;
- **trace penalty** (input-gradient regularization): $R = \frac{1}{B}\sum_i \|g_i\|^2$.

Training loss = cross-entropy + $\beta R$, fine-tuned from the same base model for 5 epochs, three values of $\beta$ for each.

**Rule, written before the run** (also in the paper plan):
- $\lambda_1$ is the right quantity if the $\lambda_1$-penalty models reach lower AS fooling than the trace-penalty models at the
  same clean accuracy (curve below, over the overlapping accuracy range);
- the defense is useful if a $\lambda_1$-penalty model has AS fooling ≤ 20% with clean accuracy ≥ 94%;
- if the curves overlap, the $\lambda_1$ claim is dropped.

First run only the second cell (the one with `drive.mount`) and press **Allow**; then *Run all* with a T4 GPU.
About 60 min. Results are saved in Google Drive (`active_subspace_runs`); after a disconnect, *Run all* again continues.

In [ ]:
%cd /content
!test -d active-subspace-bloodmnist || git clone -q https://github.com/fatemehsalmani75/active-subspace-bloodmnist.git
%cd /content/active-subspace-bloodmnist
!git pull -q
!pip install -q medmnist adversarial-robustness-toolbox

In [ ]:
import os, copy, sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from asnet.data import NUM_CLASSES, get_datasets, make_loader, subset
from asnet.models import build_model
from asnet.utils import set_seed, get_device, train_classifier, stack_dataset
from asnet.attack import as_universal_attack, fooling_rate, predict
from asnet.uap_baseline import uap_baseline
from asnet.robust import spectral_finetune
from asnet.spectrum import input_spectrum

OUT = "/content/drive/MyDrive/active_subspace_runs"
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir("/content/drive/MyDrive"):
        raise RuntimeError("Google Drive is NOT connected. Run this cell again and press Allow.")
else:
    OUT = "runs"
MODELS = os.path.join(OUT, "models")
os.makedirs(MODELS, exist_ok=True)
CSV = os.path.join(OUT, "phase2_regularization.csv")
BASE_CKPT = os.path.join(OUT, "vgg19_bloodmnist.pt")
print("Saving everything to", OUT)

set_seed(0)
device = get_device()
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print(device)

## Settings

In [ ]:
SIZE, EPOCHS, FT_EPOCHS = 64, 20, 5
DELTA, N_ATTACK, EVAL_SEEDS = 4, 200, [0, 1]
N_SPECTRUM = 2000
BETAS = {"lambda1": [1.0, 3.0, 10.0],   # lambda_1 is about 1/6 of tr C for the base model,
         "trace":   [0.1, 0.3, 1.0]}    # so these give penalties of similar size at the start

## Data and base model (the same base as Phase 1)

In [ ]:
train_aug, _, _ = get_datasets(SIZE, augment=True)
train, val, test = get_datasets(SIZE)
train_loader = make_loader(train_aug, 128, shuffle=True)

base = build_model("vgg19", NUM_CLASSES, SIZE)
if os.path.exists(BASE_CKPT):
    print("loading the base model saved in", OUT)
    base.load_state_dict(torch.load(BASE_CKPT, map_location="cpu"))
else:
    print("no base model in Drive: training one (about 7 min)")
    base, _ = train_classifier(base, train_loader, make_loader(val, 256), epochs=EPOCHS, device=device)
    torch.save(base.state_dict(), BASE_CKPT)
base.to(device).eval()

Xte, yte = stack_dataset(test, device=device)
spec_loader = make_loader(subset(train, N_SPECTRUM, seed=10), 100)

## Train and evaluate the six regularized models

Each model and each result row is saved as soon as it is ready.

In [ ]:
results = pd.read_csv(CSV).to_dict("records") if os.path.exists(CSV) else []
done = {r["model"] for r in results}

for mode, betas in BETAS.items():
    for beta in betas:
        name = f"{mode} beta={beta:g}"
        path = os.path.join(MODELS, f"phase2_{mode}_beta{beta:g}.pt")
        if name in done:
            print(f"{name}: already evaluated"); continue
        m = build_model("vgg19", NUM_CLASSES, SIZE)
        if os.path.exists(path):
            print(f"{name}: loading from Drive")
            ck = torch.load(path, map_location="cpu"); m.load_state_dict(ck["state_dict"]); t = ck["time"]
        else:
            print(f"{name}: training")
            set_seed(1)
            m, t = spectral_finetune(copy.deepcopy(base), train_loader, beta, mode=mode, epochs=FT_EPOCHS, device=device)
            torch.save({"state_dict": m.state_dict(), "time": t}, path)
        m.to(device).eval()
        for p in m.parameters():
            p.requires_grad_(False)
        s = input_spectrum(m, spec_loader, device=device)
        fa, fu = [], []
        for seed in EVAL_SEEDS:
            Xs, ys = stack_dataset(subset(train, N_ATTACK, seed=500 + seed), device=device)
            fa.append(fooling_rate(m, Xte, as_universal_attack(m, Xs, ys, delta=DELTA, seed=seed, log=lambda *a: None).v))
            fu.append(fooling_rate(m, Xte, uap_baseline(m, Xs, NUM_CLASSES, delta=DELTA, device=device)))
        row = {"model": name, "penalty": mode, "beta": beta, "train time (s)": t,
               "clean acc": (predict(m, Xte) == yte).float().mean().item(),
               "lambda1": s["lambda1"], "trace": s["trace"], "lambda1_share": s["lambda1_share"],
               "AS fooling": np.mean(fa), "AS fooling std": np.std(fa, ddof=1),
               "UAP fooling": np.mean(fu), "UAP fooling std": np.std(fu, ddof=1)}
        results.append(row)
        pd.DataFrame(results).to_csv(CSV, index=False)
        print({k: round(v, 4) if isinstance(v, float) else v for k, v in row.items()})
        del m; torch.cuda.empty_cache()

res = pd.DataFrame(results)
res.round(4)

## The decision

In [ ]:
lam = res[res.penalty == "lambda1"].sort_values("clean acc")
tra = res[res.penalty == "trace"].sort_values("clean acc")

lo = max(lam["clean acc"].min(), tra["clean acc"].min())
hi = min(lam["clean acc"].max(), tra["clean acc"].max())
if hi > lo:
    grid = np.linspace(lo, hi, 20)
    diff = np.interp(grid, lam["clean acc"], lam["AS fooling"]) - np.interp(grid, tra["clean acc"], tra["AS fooling"])
    if (diff < 0).all():
        part1 = f"lambda_1 curve is BELOW the trace curve (by {-diff.mean() * 100:.1f} points on average): lambda_1 is the right quantity."
    elif (diff > 0).all():
        part1 = f"lambda_1 curve is ABOVE the trace curve (by {diff.mean() * 100:.1f} points): the trace penalty is better."
    else:
        part1 = "The curves CROSS / OVERLAP: the lambda_1 claim is dropped; vulnerability follows the overall gradient size."
else:
    part1 = (f"No overlap in clean accuracy (lambda_1 models {lam['clean acc'].min():.3f}-{lam['clean acc'].max():.3f}, "
             f"trace models {tra['clean acc'].min():.3f}-{tra['clean acc'].max():.3f}): compare the points in the plot; no claim either way.")

useful = lam[(lam["AS fooling"] <= 0.20) & (lam["clean acc"] >= 0.94)]
part2 = ("USEFUL: " + ", ".join(f"{r.model} (fooling {r['AS fooling']:.3f}, clean {r['clean acc']:.3f})" for _, r in useful.iterrows())
         if len(useful) else "NOT USEFUL: no lambda_1 model has AS fooling <= 20% with clean accuracy >= 94%.")
print(part1); print(part2)
open(os.path.join(OUT, "phase2_decision.txt"), "w").write(part1 + "\n" + part2)

In [ ]:
p1 = os.path.join(OUT, "phase1_spectrum.csv")
base_pts = pd.read_csv(p1).set_index("model") if os.path.exists(p1) else None

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
ax = axes[0]
ax.plot(lam["clean acc"] * 100, lam["AS fooling"] * 100, "o-", c="#3b6ea5", ms=8, label="$\\lambda_1$ penalty")
ax.plot(tra["clean acc"] * 100, tra["AS fooling"] * 100, "s--", c="#d9822b", ms=8, label="trace penalty")
for _, r in pd.concat([lam, tra]).iterrows():
    ax.annotate(f"{r.beta:g}", (r["clean acc"] * 100, r["AS fooling"] * 100), textcoords="offset points", xytext=(6, 4), fontsize=8)
if base_pts is not None:
    for nm, mk in [("none (base)", "X"), ("UAT", "^"), ("PGD-AT", "v"), ("AS fine-tuning", "D")]:
        if nm in base_pts.index:
            ax.scatter(base_pts.loc[nm, "clean acc"] * 100, base_pts.loc[nm, "AS fooling"] * 100, marker=mk, s=70, c="#777", zorder=3)
            ax.annotate(nm, (base_pts.loc[nm, "clean acc"] * 100, base_pts.loc[nm, "AS fooling"] * 100), textcoords="offset points", xytext=(6, -10), fontsize=8, color="#555")
ax.axhline(20, c="gray", ls=":", lw=1); ax.axvline(94, c="gray", ls=":", lw=1)
ax.set(xlabel="clean accuracy (%)", ylabel="AS universal fooling (%)  (lower = better)", title="accuracy vs universal robustness")
ax.legend(fontsize=8)

ax = axes[1]
ax.scatter(lam["lambda1"], lam["AS fooling"] * 100, c="#3b6ea5", s=60, label="$\\lambda_1$ penalty")
ax.scatter(tra["lambda1"], tra["AS fooling"] * 100, c="#d9822b", marker="s", s=60, label="trace penalty")
if base_pts is not None:
    ax.scatter(base_pts["lambda1"], base_pts["AS fooling"] * 100, c="#777", marker="x", s=60, label="Phase 1 models")
ax.set_xscale("log")
ax.set(xlabel="$\\lambda_1$", ylabel="AS universal fooling (%)", title="all 12 models: top eigenvalue vs fooling")
ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT, "phase2_regularization.png"), bbox_inches="tight"); plt.show()